# Overall
- First try of the feature engineering was better than the second 
- Fisrt try gave approximately 90 RMSE score
- Second try gave approximately 184 RMSE score

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Upload datasets
train_df = pd.read_csv('data/train_preprocessed.csv')
test_df = pd.read_csv('data/test_preprocessed.csv')

# Detailed / First Try 

In [3]:
# Calcualting the software score
df1_train = train_df.copy()


In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Tarih/yaş ilişkili basit feature'lar
    df["years_since_graduation"] = df["application_year"] - df["graduation_year"]
    df["age_at_graduation"] = df["age"] - df["years_since_graduation"]
    df["is_recent_graduate"] = (df["years_since_graduation"] <= 1).astype(int)

    # Teknik skor özetleri
    technical_skill_cols = [
        "coding_score", "problem_solving_score", "data_structures_score",
        "sql_score", "machine_learning_score", "backend_score",
        "frontend_score", "cloud_score", "devops_score"
    ]

    df["technical_skill_mean"] = df[technical_skill_cols].mean(axis=1)
    df["technical_skill_std"] = df[technical_skill_cols].std(axis=1)
    df["technical_skill_min"] = df[technical_skill_cols].min(axis=1)
    df["technical_skill_max"] = df[technical_skill_cols].max(axis=1)
    df["technical_skill_range"] = df["technical_skill_max"] - df["technical_skill_min"]

    # Data/AI ve software odaklı teknik alt skorlar
    df["data_ai_score"] = df[
        [
            "sql_score",
            "machine_learning_score",
            "problem_solving_score",
            "data_structures_score",
            "coding_score"
        ]
    ].mean(axis=1)

    df["software_engineering_score"] = df[
        [
            "backend_score",
            "frontend_score",
            "cloud_score",
            "devops_score",
            "coding_score",
            "data_structures_score"
        ]
    ].mean(axis=1)

    # Soft skill özeti
    soft_skill_cols = [
        "communication_score",
        "teamwork_score",
        "leadership_score",
        "presentation_score",
        "linkedin_profile_score",
        "cv_quality_score",
        "hr_interview_score"
    ]

    df["soft_skill_mean"] = df[soft_skill_cols].mean(axis=1)
    df["soft_skill_std"] = df[soft_skill_cols].std(axis=1)

    # Deneyim/aktivite özeti
    experience_cols = [
        "real_client_project_count",
        "internship_count",
        "freelance_project_count",
        "hackathon_count",
        "certification_count",
        "bootcamp_count"
    ]

    df["experience_total"] = df[experience_cols].sum(axis=1)

    df["project_portfolio_score"] = (
        df["project_quality_score"]
        + df["portfolio_score"]
        + df["real_client_project_count"] * 5
        + df["freelance_project_count"] * 3
        + df["github_repo_count"] * 0.5
        + df["github_avg_stars"] * 0.5
        + df["open_source_contribution_count"] * 2
    )

    df["experience_score"] = (
        df["internship_count"] * 10
        + df["internship_duration_months"] * 2
        + df["freelance_project_count"] * 5
        + df["real_client_project_count"] * 7
    )

    df["competition_score"] = (
        df["hackathon_count"] * 3
        + df["hackathon_awards"] * 10
    )

    df["learning_activity_score"] = (
        df["certification_count"] * 2
        + df["bootcamp_count"] * 5
    )

    # Başvuru -> mülakat oranı
    df["interview_conversion_rate"] = (
        df["interviews_attended"] / (df["applications_sent"] + 1)
    )

    df["applications_without_interview"] = (
        df["applications_sent"] - df["interviews_attended"]
    )

    df["applications_per_year_after_grad"] = (
        df["applications_sent"] / (df["years_since_graduation"] + 1)
    )

    df["interview_per_year_after_grad"] = (
        df["interviews_attended"] / (df["years_since_graduation"] + 1)
    )

    df["interview_score_mean"] = df[
        ["technical_interview_score", "hr_interview_score"]
    ].mean(axis=1)

    df["weighted_interview_score"] = (
        df["technical_interview_score"] * 0.6
        + df["hr_interview_score"] * 0.4
    )

    df["interview_success_score"] = (
        df["interview_conversion_rate"] * df["weighted_interview_score"]
    )

    # Interaction feature'lar
    df["technical_x_project"] = (
        df["technical_skill_mean"] * df["project_portfolio_score"]
    )

    df["technical_x_experience"] = (
        df["technical_skill_mean"] * df["experience_score"]
    )

    df["soft_x_interview"] = (
        df["soft_skill_mean"] * df["weighted_interview_score"]
    )

    df["github_impact_score"] = (
        df["github_repo_count"]
        + df["github_avg_stars"] * 2
        + df["open_source_contribution_count"] * 3
    )

    df["internship_months_per_internship"] = (
        df["internship_duration_months"] / (df["internship_count"] + 1)
    )

    # Text'i modele direkt vermek yerine başlangıç için uzunluk feature'ları alıyoruz.
    # CatBoost text feature destekliyor ama localde daha yavaş çalışabilir.
    df["mentor_feedback_len"] = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    df["mentor_feedback_word_count"] = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.split()
        .str.len()
    )

    # Sonsuz değer temizliği
    df = df.replace([np.inf, -np.inf], np.nan)

    return df


In [5]:
df_fe = add_features(df1_train)

new_features = [
    "years_since_graduation",
    "age_at_graduation",
    "is_recent_graduate",
    "technical_skill_mean",
    "technical_skill_std",
    "technical_skill_min",
    "technical_skill_max",
    "technical_skill_range",
    "data_ai_score",
    "software_engineering_score",
    "soft_skill_mean",
    "soft_skill_std",
    "experience_total",
    "project_portfolio_score",
    "experience_score",
    "competition_score",
    "learning_activity_score",
    "interview_conversion_rate",
    "applications_without_interview",
    "applications_per_year_after_grad",
    "interview_per_year_after_grad",
    "interview_score_mean",
    "weighted_interview_score",
    "interview_success_score",
    "technical_x_project",
    "technical_x_experience",
    "soft_x_interview",
    "github_impact_score",
    "internship_months_per_internship",
    "mentor_feedback_len",
    "mentor_feedback_word_count"
]

df_train_final = df_fe[new_features + ["career_success_score"]]
print(df_train_final.head())

   years_since_graduation  age_at_graduation  is_recent_graduate  \
0                       0                 21                   1   
1                       0                 20                   1   
2                       0                 28                   1   
3                       1                 21                   1   
4                       0                 22                   1   

   technical_skill_mean  technical_skill_std  technical_skill_min  \
0             70.147169             9.815953             52.91000   
1             62.227270            11.118139             37.45074   
2             84.851461            10.685677             62.22000   
3             77.050698            12.974625             54.08000   
4             83.163333             7.417122             70.70000   

   technical_skill_max  technical_skill_range  data_ai_score  \
0                84.98               32.07000      72.810000   
1                78.90               41.44926   

# General / Second Try 

In [6]:
def feature_enginerring(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Academic Features
    df["academic_index"] = (
        df["cgpa"] * 0.5  +
        df["english_exam_score"] * 0.2 + 
        df["attendance_rate"] * 0.1 + 
        (5 - df["university_tier"]) * 0.1 # University quality (better if the tier is lower)
        - df["failed_courses_count"] * 0.1 # Penalize failed courses
    )

    # Technical Skills
    technical_cols = [
        "coding_score", "problem_solving_score", "data_structures_score",
        "sql_score", "machine_learning_score", "backend_score",
        "frontend_score", "cloud_score", "devops_score"
    ]
    df["technical_skill_avg"] = df[technical_cols].mean(axis=1)
    df["technical_skill_balance"] = df[technical_cols].std(axis=1) # More balanced skills might be better

    # Soft Skills
    soft_cols = ["communication_score", "teamwork_score", "leadership_score", "presentation_score"]
    df["soft_skill_index"] = df[soft_cols].mean(axis=1) # Average soft skill score

    # Experience
    df["internship_quality_score"] = (
        df["internship_duration_months"] * 0.7 +
        df["internship_count"] * 0.3
    )
    df["hackathon_success_rate"] = df["hackathon_awards"] / (df["hackathon_count"] + 1)

    df["experience_index"] = (
        df["internship_quality_score"] * 0.5 +
        df["real_client_project_count"] * 0.3 +
        df["freelance_project_count"] * 0.2 +
        df["hackathon_success_rate"] * 0.1
    )

    # Visibility / Portfolio
    df["github_impact_score"] = (
        df["github_repo_count"] * 0.5 +
        df["github_avg_stars"] * 0.2 +
        df["open_source_contribution_count"] * 0.3
    )

    df["visibility_index"] = (
        df["linkedin_profile_score"] * 0.4 +
        df["portfolio_score"] * 0.3 +
        df["github_impact_score"] * 0.3
    )

    # Interview Performance
    df["application_success_rate"] = df["interviews_attended"] / (df["applications_sent"] + 1)
    df["interview_score_index"] = (
        df["technical_interview_score"] * 0.6 +
        df["hr_interview_score"] * 0.4
    ) 

    return df


In [7]:
df_feature_engineered = feature_enginerring(df1_train)


# Yeni feature kolonlarını seç
new_features = [
    "academic_index", "technical_skill_avg", "technical_skill_balance",
    "soft_skill_index", "internship_quality_score", "hackathon_success_rate",
    "experience_index", "github_impact_score", "visibility_index",
    "application_success_rate", "interview_score_index"
]

# Sadece yeni feature’lar + hedef değişkeni tut
df2_train_final = df_feature_engineered[new_features + ["career_success_score"]]
